Overview
- Classify real estate search queries into language-based intent categories such as browsing, researching, and high-intent inquiry. This module focuses on NLP-based query understanding using the wording and structure of the query itself — not behavioral buyer analytics or CRM conversion data. 

Key Deliverables
- Labeled dataset: 200+ real estate search queries with intent labels ✅
- QueryIntentClassifier using logistic regression or lightweight transformer models ✅
- 80%+ accuracy on held-out test set ✅
- Confidence scores for uncertain classifications ✅
- Integration with Week 4 query parser for richer query understanding ✅
- Documentation explaining limitations of language-based intent inference ✅

Conceptually, when the user makes a query, there are three different categories of the query (i.e. browsing, researching, and high intent). From classifying the intent of the query, it will evoke Week 4 Query Parser and extract structured fields to then route to appropriate system. 

In [1]:
import ollama
import json
from pathlib import Path
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score
from sklearn.metrics import classification_report
from sklearn.metrics import confusion_matrix
import os
import sys

In [2]:
project_root = os.path.abspath('../')
if project_root not in sys.path:
    sys.path.insert(0, project_root)

from scripts.w4_queryparser import QueryParser

In [3]:
response = ollama.chat(
    model='llama3.2',
    messages=[{'role': 'user', 'content': 'Say hello in one sentence.'}]
)
print(response['message']['content'])

Hello!


Deliverable 1: 200+ labeled queries 
- Can create a 225 queries with 75 browsing, 75 researching, and 75 high-intent using ollama

In [4]:
model = "llama3.2"
OUTPUT_PATH = Path("../data/query_intent_dataset.json")

def generate_queries(intent, count=5):
    """Generate real-estate search queries for one intent category."""

    prompt = f"""
Generate {count} realistic real-estate user search queries.

All queries must have the intent: "{intent}".

Intent definitions:

BROWSING:
The user is generally exploring available properties.
They want to see or discover homes matching general preferences.
Examples:
- show me homes in San Diego
- homes with pools in Irvine
- luxury homes in Beverly Hills

RESEARCHING:
The user is trying to learn, investigate, compare, or understand 
real-estate information, markets, neighborhoods, costs, or trends.
Examples:
- areas in San Diego with low property taxes
- what is the average price per square foot in Irvine?
- how does the Irvine market compare with Anaheim?


HIGH_INTENT_INQUIRY:
The wording indicates a specific, actionable, or time-sensitive
property search.
Examples:
- move-in ready homes in San Diego under $1.2 million
- homes available this weekend with open houses
- new listings in Irvine under $900k with seller financing

For this request:
- Generate exactly {count} different queries.
- Every query must match "{intent}".
- Do not generate queries belonging to another intent.
- Make the queries realistic and varied.
- Do not repeat or create trivial variations.
- Do not mention the intent category in the query.

Return only JSON in this format:
{{
  "queries": [
    {{
      "query": "example query",
      "intent": "{intent}"
    }}
  ]
}}
"""
    try:
        response = ollama.chat(
            model=model,
            messages=[
                {
                    "role": "user",
                    "content": prompt
                }
            ],
            format={
                "type": "object",
                "properties": {
                    "queries": {
                        "type": "array",
                        "items": {
                            "type": "object",
                            "properties": {
                                "query": {
                                    "type": "string"
                                },
                                "intent": {
                                    "type": "string"
                                }
                            },
                            "required": ["query", "intent"]
                        }
                    }
                },
                "required": ["queries"]
            }
        )
        content = response["message"]["content"]

        print("\nRAW OLLAMA RESPONSE:")
        print(content)

        parsed = json.loads(content)

        print("\nPARSED TYPE:", type(parsed))

        if "queries" not in parsed:
            print("ERROR: Ollama response does not contain 'queries'.")
            return []

        queries = parsed["queries"]

        if not isinstance(queries, list):
            print("ERROR: A query item is not a list.")
            return []

        if len(queries) != count:
            print(
                f"ERROR: Expected {count} queries, "
                f"but Ollama generated {len(queries)}."
            )
            return []
        
        for item in queries:
            if not isinstance(item, dict):
                print("ERROR: A query item is not a JSON object.")
                return []

            if "query" not in item or "intent" not in item:
                print("ERROR: Query object is missing 'query' or 'intent'.")
                return []

            if item["intent"] != intent:
                print(
                    f"ERROR: Expected itent '{intent}' "
                    f"but got '{item['intent']}'."
                )
                return []
            
        return queries

    except Exception as e:
        print(f"Error generating {intent} queries: {e}")
        return []



In [5]:
def generate_intent_dataset(intent, batches=15, batch_size=5):
    """Generate 75 queries for one intent using 15 batches of 5."""

    all_queries = []

    for batch_num in range(1, batches + 1):

        print(f"\n{intent}: Batch {batch_num}/{batches}")

        queries = generate_queries(intent, count=batch_size)

        if not queries:
            print(f"Batch {batch_num} failed. Skipping.")
            continue

        all_queries.extend(queries)

        print(
            f"Batch {batch_num} added {len(queries)} queries. "
            f"Total for {intent}: {len(all_queries)}"
        )

    return all_queries

In [6]:
'''if __name__ == "__main__":
    test_queries = generate_queries("browsing", count=5)

    print("\nTEST RESULTS:")
    print(f"Number of queries: {len(test_queries)}")

    for item in test_queries:
        print(item)'''

'if __name__ == "__main__":\n    test_queries = generate_queries("browsing", count=5)\n\n    print("\nTEST RESULTS:")\n    print(f"Number of queries: {len(test_queries)}")\n\n    for item in test_queries:\n        print(item)'

In [7]:
def main():

    all_queries = []

    intents = [
        "browsing",
        "researching",
        "high_intent_inquiry"
    ]

    for intent in intents:

        print(f"\n{'=' * 50}")
        print(f"Generating {intent} queries")
        print(f"{'=' * 50}")

        queries = generate_intent_dataset(
            intent,
            batches=15,
            batch_size=5
        )

        print(
            f"\nCompleted {intent}: "
            f"{len(queries)} queries"
        )

        all_queries.extend(queries)

    # Create data directory if it doesn't exist
    OUTPUT_PATH.parent.mkdir(parents=True, exist_ok=True)

    # Save all 225 queries
    with open(OUTPUT_PATH, "w", encoding="utf-8") as f:
        json.dump(
            all_queries,
            f,
            indent=2,
            ensure_ascii=False
        )

    print("\n" + "=" * 50)
    print("DATASET GENERATION COMPLETE")
    print("=" * 50)

    print(f"Total queries: {len(all_queries)}")
    print(f"Saved to: {OUTPUT_PATH}")

In [8]:
'''if __name__ == "__main__":
    main()''' # Accomplished

'if __name__ == "__main__":\n    main()'

In [3]:
with open("../data/query_intent_dataset.json", "r", encoding="utf-8") as f:
    data = json.load(f)

print(f"Total queries: {len(data)}")

for item in data[:5]:
    print(item)

Total queries: 225
{'query': 'homes for sale in Santa Monica with ocean views', 'intent': 'browsing'}
{'query': 'affordable luxury homes in Denver', 'intent': 'browsing'}
{'query': 'new construction homes in Phoenix under $600k', 'intent': 'browsing'}
{'query': 'best neighborhoods to live in Sacramento for families', 'intent': 'browsing'}
{'query': 'rental properties in Seattle with private parking', 'intent': 'browsing'}


In [4]:
df = pd.DataFrame(data)
df.shape

(225, 2)

In [5]:
df.columns

Index(['query', 'intent'], dtype='object')

In [6]:
df['intent'].value_counts()

intent
researching            103
browsing                72
high_intent_inquiry     50
Name: count, dtype: int64

In [11]:
X = df["query"]
y = df["intent"] # classification: browsing, researching, and high intent

X_train, X_test, y_train, y_test = train_test_split(
    X, 
    y, 
    test_size=0.2,
    random_state=42, 
    stratify=y
)

In [12]:
class QueryIntentClassifier: 
    def __init__(self): 
        self.vectorizer = TfidfVectorizer(max_features=500) 
        self.model = LogisticRegression(max_iter=1000) 
        self.labels = [ 
            'browsing', 
            'researching', 
            'high_intent_inquiry' 
        ] 

    def train(self, queries, labels): 
        X = self.vectorizer.fit_transform(queries) 
        self.model.fit(X, labels) 

    def predict(self, query): 
        X = self.vectorizer.transform([query]) 
        probas = self.model.predict_proba(X)[0] 
        intent = self.model.classes_[probas.argmax()] 
        confidence = probas.max() 

        return intent, confidence

In [13]:
classifier = QueryIntentClassifier()

In [14]:
classifier.train(X_train, y_train)

In [10]:
results = [
    classifier.predict(query)
    for query in X_test
]

predictions = [result[0] for result in results]
confidences = [result[1] for result in results]

accuracy = accuracy_score(y_test, predictions)

print(f"Test Accuracy: {accuracy:.2%}")

Test Accuracy: 93.33%


Meet Deliverable 2 (test accuracy > 80%)

In [11]:
print(classification_report(
    y_test,
    predictions
))

                     precision    recall  f1-score   support

           browsing       0.82      1.00      0.90        14
high_intent_inquiry       1.00      0.70      0.82        10
        researching       1.00      1.00      1.00        21

           accuracy                           0.93        45
          macro avg       0.94      0.90      0.91        45
       weighted avg       0.95      0.93      0.93        45



In [12]:
cm = confusion_matrix(
    y_test,
    predictions,
    labels=[
        "browsing",
        "researching",
        "high_intent_inquiry"
    ]
)

print(cm)

[[14  0  0]
 [ 0 21  0]
 [ 3  0  7]]


In [13]:
results = pd.DataFrame({
    "query": X_test,
    "actual": y_test,
    "predicted": predictions
})

errors = results[
    results["actual"] != results["predicted"]
]

print(errors.to_string(index=False))

                                                                                                                     query              actual predicted
                                                           new listings in Newport Beach under $800k with seller financing high_intent_inquiry  browsing
                                          show me foreclosed homes for sale in San Francisco this weekend with open houses high_intent_inquiry  browsing
properties coming up for sale by the end of March 2024 in San Gabriel Valley with 2-3 bedrooms and a minimum of 1,500 sqft high_intent_inquiry  browsing


The Ollama model defined some queries differently than the definition given, so will validate the queries in dataset

In [14]:
results_df = pd.DataFrame({
    "query": X_test,
    "actual": y_test,
    "predicted": predictions,
    "confidence": confidences
})

In [15]:
uncertain = results_df[
    results_df["confidence"] < 0.60
]

print(uncertain.to_string(index=False))

                                                                                                                     query              actual           predicted  confidence
                                                      homes in the coastal area of Orange County for rent with ocean views            browsing            browsing    0.440350
                                                                                homes with views of the ocean in San Diego            browsing            browsing    0.594074
                                 new listings in Orange County under $900k with seller financing and short closing process high_intent_inquiry high_intent_inquiry    0.505454
                                                    explore real estate options for first-time homebuyers in Santa Barbara         researching         researching    0.543701
                                                   the cheapest houses to buy in Phoenix that are still under construction   

In [16]:
results_df.shape

(45, 4)

In [17]:
average_confidence = results_df.groupby("predicted")["confidence"].mean()

print(average_confidence)

predicted
browsing               0.567187
high_intent_inquiry    0.614371
researching            0.724932
Name: confidence, dtype: float64


In [18]:
confidence_summary = (
    results_df
    .groupby("predicted")
    .agg(
        average_confidence=("confidence", "mean"),
        number_of_predictions=("confidence", "count")
    )
)

confidence_summary["average_confidence"] *= 100

print(confidence_summary)

                     average_confidence  number_of_predictions
predicted                                                     
browsing                      56.718746                     17
high_intent_inquiry           61.437060                      7
researching                   72.493192                     21


In [19]:
results_df["confidence"].mean()

0.6481411401375256

Confidence analysis: The classifier produced an average confidence of 64.81% across the 45 held-out test queries. Confidence varied by predicted intent, with the highest average confidence for researching queries (72.49%), followed by high-intent inquiries (61.44%) and browsing queries (56.72%). The lower confidence for browsing and high-intent classifications reflects the linguistic overlap between general property exploration and more specific property searches. 

In [15]:
parser = QueryParser()

In [16]:
parsed = parser.parse(df['query'].iloc[0])

In [17]:
parsed

{'city': 'Santa Monica', 'view': True}

In [18]:
def understand_query(query):
    parsed = parser.parse(query)

    intent, confidence = classifier.predict(query)

    return {
        "parsed": parsed, 
        "intent": intent, 
        "confidence": confidence
    }
    

In [19]:
df["parsed"] = df["query"].apply(parser.parse)

In [20]:
df[["predicted_intent", "intent_confidence"]] = df["query"].apply(
    lambda q: pd.Series(classifier.predict(q))
)

In [21]:
df.head()

,query,intent,parsed,predicted_intent,intent_confidence
0,homes for sale in Santa Monica with ocean views,browsing,"{'city': 'Santa Monica', 'view': True}",browsing,0.813589
1,affordable luxury homes in Denver,browsing,{},browsing,0.714539
2,new construction homes in Phoenix under $600k,browsing,{'price_max': 600000},browsing,0.520097
3,best neighborhoods to live in Sacramento for f...,browsing,{'city': 'Sacramento'},researching,0.632777
4,rental properties in Seattle with private parking,browsing,{},browsing,0.538043


In [22]:
df['intent'].value_counts()

intent
researching            103
browsing                72
high_intent_inquiry     50
Name: count, dtype: int64

In [25]:
df['parsed'].value_counts()

parsed
{'city': 'San Diego'}                                              27
{'city': 'Irvine'}                                                 18
{'city': 'Orange'}                                                 16
{}                                                                 11
{'city': 'Los Angeles'}                                             9
                                                                   ..
{'city': 'Orange', 'view': True}                                    1
{'price_min': 1, 'price_max': 2000000, 'city': 'Beverly Hills'}     1
{'bedrooms_min': 3, 'city': 'Irvine'}                               1
{'price_max': 500000, 'city': 'Los Angeles'}                        1
{'price_max': 2500000, 'city': 'Irvine', 'view': True}              1
Name: count, Length: 110, dtype: int64

In [27]:
features = [
    "price_max",
    "price_min",
    "bedrooms",
    "bedrooms_min", 
    "bathrooms",
    "bathrooms_min",
    "city",
    "year_built_min",
    "year_built_max",
    "year_built",
    "dom_max",
    "dom_min",
    "hoa_max",
    "hoa_min",
    "sqft",
    "sqft_min",
    "sqft_max",
    "property_type",
    "view_type",
    "view",
    "fireplace",
    "pool",
    "amenity"
]

In [28]:
def has_feature(parsed, feature):
    if not isinstance(parsed, dict):
        return False

    value = parsed.get(feature)

    if value is None:
        return False

    if value is False:
        return False

    if value == "":
        return False

    return True

In [29]:
intents = [
    "browsing",
    "researching",
    "high_intent_inquiry"
]

feature_percentages = {}

for feature in features:
    feature_percentages[feature] = {}

    for intent in intents:
        intent_df = df[df["intent"] == intent]

        count_with_feature = intent_df["parsed"].apply(
            lambda x: has_feature(x, feature)
        ).sum()

        percentage = (
            count_with_feature / len(intent_df)
        ) * 100

        feature_percentages[feature][intent] = percentage

In [30]:
feature_table = pd.DataFrame(feature_percentages).T

feature_table.columns = [
    "Browsing",
    "Researching",
    "High Intent"
]

print(feature_table.round(1))

                Browsing  Researching  High Intent
price_max           23.6          3.9         58.0
price_min           13.9          1.0         16.0
bedrooms            12.5          3.9         16.0
bedrooms_min         4.2          0.0          2.0
bathrooms            8.3          0.0          8.0
bathrooms_min        1.4          0.0          4.0
city                81.9         92.2         92.0
year_built_min       0.0          0.0          0.0
year_built_max       1.4          0.0          0.0
year_built           0.0          0.0          0.0
dom_max              0.0          0.0          0.0
dom_min              0.0          0.0          0.0
hoa_max              0.0          0.0          2.0
hoa_min              0.0          0.0          0.0
sqft                 1.4          0.0          2.0
sqft_min             4.2          0.0          0.0
sqft_max             0.0          0.0          0.0
property_type        6.9          2.9          6.0
view_type            1.4       

The classification results showed that researching queries were classified more accurately than browsing and high-intent inquiry queries. This suggests that the language used in browsing and high-intent queries can sometimes be more difficult to distinguish based solely on wording. In particular, a query may contain highly specific criteria, such as price, number of bedrooms, location, or amenities, which can make it appear to represent a high-intent inquiry. However, the user may simply have a specific idea of what they want to browse for and may not actually intend to purchase a property in the near future.

Another observation was that the classifier generally produced higher confidence when predicting researching queries compared with browsing and high-intent inquiry queries. This suggests that researching queries exhibit more distinctive inquiry-oriented language patterns, which may make them easier for the classifier to distinguish from browsing and high-intent inquiries. In contrast, browsing and high-intent queries can share similar property-search language, making them more difficult for a language-only classifier to distinguish.

Another piece of evidence that shows that browsing and high-intent queries generally contained more property-specific information than researching queries is that price_max was present in 23.6% of browsing queries and 58.0% of high-intent queries, compared with only 3.9% of researching queries. Similarly, bedrooms and bathrooms appeared more frequently in browsing and high-intent queries than in research queries. The frequencies in these features between the two kinds of queries can overlap, resulting in difficulty for a language-based classifier to distinguish. 

One final limitation I would like to share is that by using Ollama to create this 200+ intent labeled dataset, the query distribution may reflect the language patterns of the model used to generate the training data rather than the language patterns of real users despite providing the definitions of these intents. The queries may lack in real-world user language despite the AI model attempting to replicate it. In other words, the AI-generated queries may not fully capture the ambiguity, variation, and natural phrasing found in real-world user searches. Also, the AI model cannot interpret the queries entirely the same as how a person may interpret them, resulting in me to validate some of the queries that were initially classified differently. 
